# Modélisation de la PD Corporate — Notebook complet

Ce notebook fournit une chaîne complète et reproductible de modélisation de la **Probability of Default (PD)** sur un périmètre **Corporate**.

Le workflow couvre :

1. Population Corporate et panel annuel
2. Definition of Default
3. Préparation Development / OOT
4. Binning monotone
5. WoE / IV
6. Corrélation / VIF
7. Régression logistique
8. AUC / Gini / KS / Brier
9. Validation Out-of-Time
10. Calibration TTC
11. Long-Run Default Rate
12. Master Scale à 13 grades
13. Lissage des PD
14. Margin of Conservatism
15. PD floor
16. Overrides
17. Backtesting binomial / Jeffreys / Vasicek
18. PSI
19. Scoring du portefeuille 2023
20. Contrôles finaux et exports

> **Important :** les données sont simulées et les éléments réglementaires (notamment calibration, MoC, corrélation Vasicek et capital) sont illustratifs. Pour une utilisation IRB réelle, ils doivent être remplacés par les méthodologies et textes applicables de l'établissement.

## 0. Environnement

Le notebook est prévu pour **Python $\geq 3.10$**.

Bibliothèques utilisées :

- `numpy`
- `pandas`
- `scipy`
- `statsmodels`
- `scikit-learn`

La seed est fixée afin de garantir la reproductibilité des résultats.

In [72]:
import sys
import warnings

assert sys.version_info >= (3, 10), "Python >= 3.10 requis"

import numpy as np
import pandas as pd

from scipy.special import expit, logit
from scipy.stats import norm, binomtest, beta, spearmanr

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

SEED = 42
rng = np.random.default_rng(SEED)

YEARS = list(range(2015, 2024))
DEVELOPMENT_YEARS = list(range(2015, 2021))
OOT_YEARS = [2021, 2022, 2023]
TARGET_YEAR = 2023

N_OBLIGORS = 3500

IV_THRESHOLD = 0.02
CORR_MAX = 0.75
VIF_MAX = 5.0
P_VALUE_MAX = 0.05

print("Python :", sys.version)
print("Pandas :", pd.__version__)
print("Période :", YEARS)

Python : 3.13.15 (v3.13.15:4061bc4c35f, Aug  5 2026, 09:08:51) [Clang 21.0.0 (clang-2100.1.1.101)]
Pandas : 3.1.0
Période : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


## 1. Fonctions générales

Les fonctions suivantes sont utilisées pour les métriques de performance et les contrôles de calibration.

Le **Gini** est calculé par :

$$
Gini = 2 \times AUC - 1
$$

La statistique **KS** mesure l'écart maximal entre les distributions cumulées des bons et des mauvais.

In [73]:
def gini_from_auc(auc):
    return 2 * auc - 1


def safe_auc(y_true, score):
    y_true = pd.Series(y_true)
    score = pd.Series(score)
    if y_true.nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, score)


def ks_stat(y_true, score):
    y_true = pd.Series(y_true).reset_index(drop=True)
    score = pd.Series(score).reset_index(drop=True)

    order = np.argsort(-score.to_numpy())
    y = y_true.to_numpy()[order]

    bad = (y == 1).astype(int)
    good = (y == 0).astype(int)

    bad_cum = bad.cumsum() / max(bad.sum(), 1)
    good_cum = good.cumsum() / max(good.sum(), 1)

    return np.max(np.abs(bad_cum - good_cum))


def performance_table(y, pd_pred):
    auc = safe_auc(y, pd_pred)
    return pd.Series({
        "AUC": auc,
        "Gini": gini_from_auc(auc) if pd.notna(auc) else np.nan,
        "KS": ks_stat(y, pd_pred),
        "Brier": brier_score_loss(y, pd_pred),
        "Observed_DR": np.mean(y),
        "Mean_PD": np.mean(pd_pred),
    })


def calibration_table(y_true, pd_pred, n_bins=10):
    df = pd.DataFrame({
        "y": np.asarray(y_true),
        "pd_pred": np.asarray(pd_pred)
    }).dropna()

    if df.empty:
        return pd.DataFrame()

    q = min(n_bins, df["pd_pred"].nunique())

    if q < 2:
        df["bin"] = 1
    else:
        df["bin"] = pd.qcut(
            df["pd_pred"],
            q=q,
            duplicates="drop",
            labels=False
        ) + 1

    out = (
        df.groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            defaults=("y", "sum"),
            observed_dr=("y", "mean"),
            mean_pd=("pd_pred", "mean"),
        )
        .reset_index()
    )

    out["ratio_dr_pd"] = (
        out["observed_dr"] /
        out["mean_pd"].replace(0, np.nan)
    )

    return out

## 2. Definition of Default

On utilise ici une définition simplifiée :

$$
Default =
\mathbf{1}
\left[
DPD > 90
\;\lor\;
\frac{Arrears}{Exposure}>1\%
\;\lor\;
UTP=1
\right]
$$

Cette règle est uniquement illustrative. En production, la définition doit être alignée avec la politique de défaut de l'établissement et le cadre réglementaire applicable.

In [74]:
def default_of_obligor(
    dpd_next12m,
    arrears_amt,
    exposure,
    utp_flag,
    dpd_threshold=90,
    arrears_threshold=0.01,
):
    exposure = np.asarray(exposure)
    arrears_amt = np.asarray(arrears_amt)

    arrears_ratio = np.divide(
        arrears_amt,
        np.maximum(exposure, 1e-9)
    )

    return (
        (np.asarray(dpd_next12m) > dpd_threshold)
        | (arrears_ratio > arrears_threshold)
        | (np.asarray(utp_flag) == 1)
    ).astype(int)

## 3. Population Corporate

On simule des obligors Corporate avec :

- secteur ;
- taille ;
- rating structurel ;
- levier ;
- marge structurelle ;
- volatilité financière.

Chaque obligor est ensuite observé sur plusieurs années.

In [75]:
obligors = pd.DataFrame({
    "obligor_id": np.arange(1, N_OBLIGORS + 1),
    "entry_year": rng.integers(2013, 2017, N_OBLIGORS),
})

obligors["sector"] = rng.choice(
    ["Industry", "Services", "Construction", "Retail", "Tech", "Energy"],
    size=N_OBLIGORS,
    p=[0.20, 0.25, 0.15, 0.15, 0.15, 0.10]
)

obligors["size_class"] = rng.choice(
    ["Small", "Medium", "Large"],
    size=N_OBLIGORS,
    p=[0.50, 0.35, 0.15]
)

obligors["base_rating"] = rng.normal(0, 1, N_OBLIGORS)

obligors["structural_leverage"] = np.clip(
    rng.lognormal(mean=-0.1, sigma=0.6, size=N_OBLIGORS),
    0.1, 8
)

obligors["structural_margin"] = np.clip(
    rng.normal(0.08, 0.06, N_OBLIGORS),
    -0.25, 0.40
)

obligors["financial_volatility"] = np.clip(
    rng.lognormal(mean=-2.0, sigma=0.5, size=N_OBLIGORS),
    0.02, 1.0
)

obligors.head()

,obligor_id,entry_year,sector,size_class,base_rating,structural_leverage,structural_margin,financial_volatility
0,1,2013,Retail,Small,1.487167,0.479330,0.073048,0.082659
1,2,2016,Services,Small,-0.770866,0.874431,0.025127,0.196545
2,3,2015,Tech,Small,-0.483588,0.812126,0.073194,0.068478
3,4,2014,Tech,Medium,2.181376,1.217971,0.152935,0.069561
4,5,2014,Services,Medium,0.096261,1.638167,-0.016882,0.135934


## 4. Construction du panel annuel

Une ligne représente une observation :

$$
(obligor,\ year)
$$

Les observations sont limitées à la période d'activité de l'entreprise.

In [76]:
panel = (
    obligors
    .assign(key=1)
    .merge(
        pd.DataFrame({"year": YEARS, "key": 1}),
        on="key"
    )
    .drop(columns="key")
)

panel = panel[
    panel["year"] >= panel["entry_year"]
].copy()

panel = panel.sort_values(
    ["obligor_id", "year"]
).reset_index(drop=True)

panel["age"] = panel["year"] - panel["entry_year"]

panel.shape

(30607, 10)

,obligor_id,entry_year,sector,size_class,base_rating,structural_leverage,structural_margin,financial_volatility,year,age,gdp_growth,macro_stress,leverage,debt_ebitda,ebitda_margin,roa,liquidity_ratio,cash_ratio,utilization,management_quality,governance_score,pd_pit_true,default_true,max_dpd_next12m,exposure,arrears_amt,utp_flag,default_dod
0,1,2013,Retail,Small,1.487167,0.479330,0.073048,0.082659,2015,2,0.025000,-0.011873,0.574211,2.574386,0.042089,-0.020254,1.354427,0.266604,0.580165,0.609282,0.862914,0.028476,0,24,"100,000.000000",534.860779,0,0
1,1,2013,Retail,Small,1.487167,0.479330,0.073048,0.082659,2016,3,0.030000,-0.031121,0.496974,2.228106,0.144167,0.075761,1.447251,0.297467,0.587885,0.950161,2.026626,0.010737,0,64,"100,000.000000",193.905234,0,0
2,1,2013,Retail,Small,1.487167,0.479330,0.073048,0.082659,2017,4,0.028000,-0.032633,0.446402,2.001376,0.093126,0.011232,1.232721,0.417952,0.341251,NaN,1.180685,0.016231,0,11,"113,868.357497",866.407927,0,0
3,1,2013,Retail,Small,1.487167,0.479330,0.073048,0.082659,2018,5,0.020000,-0.024464,0.411881,1.846606,0.092106,0.057237,1.434843,NaN,0.516528,1.976954,0.238095,0.012726,0,6,"754,613.731613","3,685.558852",0,0
4,1,2013,Retail,Small,1.487167,0.479330,0.073048,0.082659,2019,6,0.015000,-0.018410,0.500979,2.246062,0.075398,0.024488,1.443055,0.285265,0.617657,1.724400,0.425879,0.018308,0,68,"185,442.523907","1,166.062593",0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30602,3500,2014,Tech,Medium,0.366048,1.191711,0.006328,0.115040,2019,5,0.015000,-0.018410,1.225595,7.839898,-0.003992,-0.034434,0.985509,0.384734,0.526955,0.494546,-0.711065,0.079627,0,8,"404,394.614567","2,113.849977",0,0
30603,3500,2014,Tech,Medium,0.366048,1.191711,0.006328,0.115040,2020,6,-0.020000,0.009205,1.053640,6.739935,0.005456,-0.041272,NaN,0.371103,0.580249,NaN,NaN,0.069196,0,51,"100,000.000000",97.498206,0,0
30604,3500,2014,Tech,Medium,0.366048,1.191711,0.006328,0.115040,2021,7,0.035000,-0.039721,0.881647,5.639724,0.001607,NaN,1.577688,0.456651,0.545234,0.535292,0.079604,0.041158,0,15,"222,051.796032","1,746.751472",0,0
30605,3500,2014,Tech,Medium,0.366048,1.191711,0.006328,0.115040,2022,8,0.030000,-0.028823,1.177286,7.530873,0.001267,-0.031320,1.582280,0.527150,0.500431,0.174466,-0.240534,0.057126,0,70,"206,363.946657",687.169980,0,0


## 5. Variables macroéconomiques

On introduit un facteur macroéconomique afin de différencier :

- risque structurel / TTC ;
- risque conjoncturel / PIT.

Le stress macroéconomique augmente lorsque la croissance ralentit.

In [77]:
macro = pd.DataFrame({
    "year": YEARS,
    "gdp_growth": [
        0.025, 0.030, 0.028, 0.020, 0.015,
        -0.020, 0.035, 0.030, 0.022
    ]
})

macro["macro_stress"] = (
    -macro["gdp_growth"]
    + rng.normal(0, 0.01, len(macro))
)

panel = panel.merge(macro, on="year", how="left")

panel[["year", "gdp_growth", "macro_stress"]].drop_duplicates().sort_values("year")

,year,gdp_growth,macro_stress
0,2015,0.025000,-0.011873
1,2016,0.030000,-0.031121
2,2017,0.028000,-0.032633
3,2018,0.020000,-0.024464
4,2019,0.015000,-0.018410
5,2020,-0.020000,0.009205
6,2021,0.035000,-0.039721
7,2022,0.030000,-0.028823
8,2023,0.022000,-0.017956


## 6. Variables financières et qualitatives

On construit les variables candidates au modèle.

Une structure classique Corporate comprend notamment :

$$
Leverage = \frac{Debt}{Equity}
$$

$$
Debt/EBITDA = \frac{Debt}{EBITDA}
$$

ainsi que des indicateurs de rentabilité, liquidité, utilisation et qualité qualitative.

In [78]:
n = len(panel)

panel["leverage"] = np.clip(
    panel["structural_leverage"] *
    np.exp(rng.normal(0, 0.15, n)),
    0.05, 15
)

panel["debt_ebitda"] = np.clip(
    panel["leverage"] /
    (np.maximum(panel["structural_margin"], -0.10) + 0.15),
    0.1, 30
)

panel["ebitda_margin"] = np.clip(
    panel["structural_margin"]
    - 0.20 * panel["macro_stress"]
    + rng.normal(0, 0.025, n),
    -0.40, 0.50
)

panel["roa"] = np.clip(
    0.6 * panel["ebitda_margin"]
    - 0.03 * panel["leverage"]
    + rng.normal(0, 0.015, n),
    -0.50, 0.40
)

panel["liquidity_ratio"] = np.clip(
    1.5
    - 0.12 * panel["leverage"]
    + rng.normal(0, 0.20, n),
    0.05, 4
)

panel["cash_ratio"] = np.clip(
    0.40
    - 0.03 * panel["leverage"]
    + rng.normal(0, 0.08, n),
    0.01, 1.5
)

panel["utilization"] = np.clip(
    0.45
    + 0.08 * panel["leverage"]
    + 0.10 * panel["macro_stress"]
    + rng.normal(0, 0.08, n),
    0.01, 1.20
)

panel["management_quality"] = np.clip(
    panel["base_rating"] + rng.normal(0, 0.50, n),
    -3, 3
)

panel["governance_score"] = np.clip(
    0.5 * panel["base_rating"]
    + rng.normal(0, 0.70, n),
    -3, 3
)

candidate_vars = [
    "leverage",
    "debt_ebitda",
    "ebitda_margin",
    "roa",
    "liquidity_ratio",
    "cash_ratio",
    "utilization",
    "management_quality",
    "governance_score",
]

candidate_vars

['leverage',
 'debt_ebitda',
 'ebitda_margin',
 'roa',
 'liquidity_ratio',
 'cash_ratio',
 'utilization',
 'management_quality',
 'governance_score']

## 7. Données manquantes

Les données Corporate peuvent présenter des taux de missingness importants.

Le notebook traite les valeurs manquantes comme une modalité distincte lors de la transformation WoE.

In [79]:
missing_rates = {
    "debt_ebitda": 0.06,
    "roa": 0.04,
    "liquidity_ratio": 0.05,
    "cash_ratio": 0.08,
    "management_quality": 0.10,
    "governance_score": 0.12,
}

for col, rate in missing_rates.items():
    mask = rng.random(len(panel)) < rate
    panel.loc[mask, col] = np.nan

panel.isna().mean().sort_values(ascending=False).head(15)

governance_score       0.121149
management_quality     0.100304
cash_ratio             0.077499
debt_ebitda            0.058549
liquidity_ratio        0.051491
roa                    0.039272
macro_stress           0.000000
utilization            0.000000
ebitda_margin          0.000000
leverage               0.000000
obligor_id             0.000000
entry_year             0.000000
age                    0.000000
year                   0.000000
financial_volatility   0.000000
dtype: float64

## 8. Risque latent et simulation du défaut

Le risque latent est construit sous forme logistique :

$$
logit(PD_{PIT})
=
\beta_0
+
\sum_j \beta_j X_j
+
\beta_M M
$$

puis :

$$
PD_{PIT}
=
\frac{1}{1+e^{-logit(PD)}}
$$

Le défaut est ensuite tiré selon cette probabilité.

In [80]:
latent_score = (
    -3.9
    + 0.55 * np.log1p(panel["leverage"])
    + 0.35 * np.log1p(panel["debt_ebitda"].clip(lower=0))
    - 4.00 * panel["ebitda_margin"].fillna(panel["ebitda_margin"].median())
    - 1.50 * panel["roa"].fillna(panel["roa"].median())
    - 0.35 * panel["liquidity_ratio"].fillna(panel["liquidity_ratio"].median())
    - 0.25 * panel["cash_ratio"].fillna(panel["cash_ratio"].median())
    + 1.20 * panel["utilization"]
    - 0.25 * panel["management_quality"].fillna(panel["management_quality"].median())
    - 0.20 * panel["governance_score"].fillna(panel["governance_score"].median())
    + 1.50 * panel["macro_stress"]
)

panel["pd_pit_true"] = np.clip(
    expit(latent_score),
    0.0001,
    0.80
)

panel["default_true"] = (
    rng.random(len(panel))
    < panel["pd_pit_true"]
).astype(int)

panel["pd_pit_true"].describe()

count   28,815.000000
mean         0.046373
std          0.038191
min          0.002697
25%          0.022551
50%          0.035307
75%          0.056672
max          0.496347
Name: pd_pit_true, dtype: float64

## 9. Génération des événements observables de défaut

Nous simulons :

- le nombre maximal de jours de retard à 12 mois ;
- le montant des arriérés ;
- l'exposition ;
- l'indicateur UTP.

Ces informations permettront de reconstruire la variable de défaut sans utiliser directement le défaut latent.

In [81]:
default_mask = panel["default_true"].eq(1)

panel["max_dpd_next12m"] = np.where(
    default_mask,
    rng.integers(91, 181, len(panel)),
    rng.integers(0, 91, len(panel))
)

panel["exposure"] = np.clip(
    np.exp(
        rng.normal(
            np.where(panel["size_class"].eq("Large"), 14, 12),
            0.8,
            len(panel)
        )
    ),
    100_000,
    500_000_000
)

panel["arrears_amt"] = np.where(
    default_mask,
    panel["exposure"] * rng.uniform(0.02, 0.40, len(panel)),
    panel["exposure"] * rng.uniform(0, 0.008, len(panel))
)

panel["utp_flag"] = np.where(
    default_mask,
    rng.random(len(panel)) < 0.45,
    rng.random(len(panel)) < 0.005
).astype(int)

panel["default_dod"] = default_of_obligor(
    panel["max_dpd_next12m"],
    panel["arrears_amt"],
    panel["exposure"],
    panel["utp_flag"]
)

pd.crosstab(
    panel["default_true"],
    panel["default_dod"],
    rownames=["default_true"],
    colnames=["default_dod"]
)

default_dod,0,1
default_true,,
0,29050,148
1,0,1409


## 10. Suppression des observations postérieures au défaut

Après le premier défaut, les observations suivantes ne doivent pas être considérées comme des observations normales indépendantes pour le développement d'une PD.

On conserve donc les observations jusqu'au premier défaut inclus.

In [82]:
first_default = (
    panel.loc[panel["default_dod"].eq(1)]
    .groupby("obligor_id")["year"]
    .min()
    .rename("first_default_year")
)

panel_model = panel.merge(
    first_default,
    on="obligor_id",
    how="left"
)

panel_model = panel_model[
    panel_model["first_default_year"].isna()
    | (
        panel_model["year"]
        <= panel_model["first_default_year"]
    )
].copy()

panel_model.shape

(25490, 29)

## 11. Development / OOT

Le développement porte sur 2015–2020.

La validation OOT porte sur 2021–2023.

Cette séparation protège contre le risque de sur-apprentissage temporel.

In [83]:
target = "default_dod"

dev = panel_model[
    panel_model["year"].isin(DEVELOPMENT_YEARS)
].copy()

oot = panel_model[
    panel_model["year"].isin(OOT_YEARS)
].copy()

print("Development :", dev.shape)
print("OOT :", oot.shape)
print("DR Development :", dev[target].mean())
print("DR OOT :", oot[target].mean())

Development : (17926, 29)
OOT : (7564, 29)
DR Development : 0.04858864219569341
DR OOT : 0.04362771020624009


## 12. Winsorisation

Les ratios financiers peuvent présenter des observations extrêmes.

On applique une winsorisation 1% / 99% calculée **uniquement sur le développement**, puis réutilisée sur l'OOT.

In [84]:
def fit_winsor_limits(df, cols, lower=0.01, upper=0.99):
    limits = {}
    for c in cols:
        limits[c] = (
            df[c].quantile(lower),
            df[c].quantile(upper)
        )
    return limits


def apply_winsor_limits(df, limits):
    out = df.copy()
    for c, (lo, hi) in limits.items():
        out[c] = out[c].clip(lo, hi)
    return out


winsor_limits = fit_winsor_limits(dev, candidate_vars)

dev_w = apply_winsor_limits(dev, winsor_limits)
oot_w = apply_winsor_limits(oot, winsor_limits)

## 13. Binning monotone

Les classes sont construites de façon à rechercher une relation monotone entre la variable et le taux de défaut.

Le principe est :

$$
DR_{Bin_1}
\leq
DR_{Bin_2}
\leq
...
\leq
DR_{Bin_K}
$$

ou l'inverse selon le sens économique de la variable.

In [89]:

import numpy as np
import pandas as pd

def monotonic_bins(x, y, max_bins=5, min_pct=0.05, max_iter=20):
    """
    Création de classes numériques à partir des quantiles.
    Gère les variables constantes, les valeurs manquantes
    et les faibles nombres de valeurs distinctes.
    """

    s = pd.DataFrame({
        "x": pd.to_numeric(x, errors="coerce"),
        "y": pd.to_numeric(y, errors="coerce")
    })

    # Suppression des observations sans x ou sans y
    s = s.dropna(subset=["x", "y"])

    # Vérification des données disponibles
    if s.empty:
        return np.array([-np.inf, np.inf])

    # Vérification de la cardinalité
    n_unique = s["x"].nunique()

    # Variable constante : une seule classe
    if n_unique <= 1:
        return np.array([-np.inf, np.inf])

    # Nombre maximal de classes adapté aux données
    n_bins = min(max_bins, n_unique)

    # Calcul des quantiles
    quantiles = np.linspace(0, 1, n_bins + 1)

    edges = (
        s["x"]
        .quantile(quantiles)
        .drop_duplicates()
        .to_numpy(dtype=float)
    )

    # Si les quantiles ne permettent pas de créer plusieurs classes
    if len(edges) <= 2:
        return np.array([-np.inf, np.inf])

    # Construction des bornes
    edges = np.concatenate((
        [-np.inf],
        edges[1:-1],
        [np.inf]
    ))

    # Suppression des bornes dupliquées
    edges = np.unique(edges)

    # Vérification finale
    if len(edges) < 3:
        return np.array([-np.inf, np.inf])

    return edges


## 14. WoE / IV

Convention retenue :

$$
WoE_i =
\ln
\left(
\frac{\%Bad_i}{\%Good_i}
\right)
$$

L'Information Value est :

$$
IV =
\sum_i
(\%Bad_i-\%Good_i)
\times WoE_i
$$

Le WoE est calculé avec un léger lissage afin d'éviter les divisions par zéro.

In [90]:
class WoEBinner:
    def __init__(self, numeric_cols=None):
        self.numeric_cols = numeric_cols or []
        self.edges = {}
        self.maps = {}
        self.iv = {}

    def _assign_numeric(self, x, edges):
        return (
            pd.cut(x, bins=edges, include_lowest=True)
            .astype("string")
            .fillna("MISSING")
        )

    def fit(self, X, y):
        y = pd.Series(y).astype(int)

        total_good = max((y == 0).sum(), 1)
        total_bad = max((y == 1).sum(), 1)

        for c in X.columns:
            if c in self.numeric_cols:
                edges = monotonic_bins(X[c], y)
                self.edges[c] = edges
                cats = self._assign_numeric(X[c], edges)
            else:
                cats = X[c].astype("string").fillna("MISSING")

            tmp = pd.DataFrame({"cat": cats, "y": y})

            agg = (
                tmp.groupby("cat", dropna=False)
                .agg(
                    good=("y", lambda z: (z == 0).sum()),
                    bad=("y", lambda z: (z == 1).sum())
                )
            )

            good_dist = (
                agg["good"] + 0.5
            ) / (
                total_good + 0.5 * len(agg)
            )

            bad_dist = (
                agg["bad"] + 0.5
            ) / (
                total_bad + 0.5 * len(agg)
            )

            woe = np.log(bad_dist / good_dist)

            iv = (
                (bad_dist - good_dist) * woe
            ).sum()

            self.maps[c] = woe.to_dict()
            self.iv[c] = float(iv)

        return self

    def transform(self, X):
        out = pd.DataFrame(index=X.index)

        for c in X.columns:
            if c in self.numeric_cols:
                cats = self._assign_numeric(X[c], self.edges[c])
            else:
                cats = X[c].astype("string").fillna("MISSING")

            out[c] = (
                cats.map(self.maps[c])
                .fillna(0.0)
                .astype(float)
            )

        return out


binner = WoEBinner(numeric_cols=candidate_vars)

binner.fit(
    dev_w[candidate_vars],
    dev_w[target]
)

iv_table = (
    pd.Series(binner.iv, name="IV")
    .sort_values(ascending=False)
)

iv_table

debt_ebitda          0.469853
roa                  0.213395
leverage             0.177385
ebitda_margin        0.127396
utilization          0.112008
management_quality   0.101661
governance_score     0.082896
liquidity_ratio      0.045162
cash_ratio           0.014032
Name: IV, dtype: float64

## 15. Sélection par IV

On conserve les variables présentant un IV supérieur au seuil.

Ce filtre constitue uniquement une présélection ; il ne remplace pas les analyses économiques, statistiques et de stabilité.

In [91]:
selected_iv = iv_table[
    iv_table >= IV_THRESHOLD
].index.tolist()

print("Variables après IV :")
print(selected_iv)

Variables après IV :
['debt_ebitda', 'roa', 'leverage', 'ebitda_margin', 'utilization', 'management_quality', 'governance_score', 'liquidity_ratio']


## 16. Transformation WoE

Les paramètres sont appris sur le développement et appliqués tels quels à l'OOT.

In [92]:
W_dev = binner.transform(
    dev_w[selected_iv]
)

W_oot = binner.transform(
    oot_w[selected_iv]
)

W_dev.head()

,debt_ebitda,roa,leverage,ebitda_margin,utilization,management_quality,governance_score,liquidity_ratio
0,-0.406768,0.654433,-0.413202,0.113242,0.129433,-0.162582,-0.526196,-0.006806
1,-0.668571,-0.665548,-0.554380,-0.544097,0.129433,-0.162582,-0.526196,-0.164200
2,-0.668571,-0.137010,-0.554380,-0.101995,-0.564471,-0.195644,-0.526196,0.049559
3,-0.668571,-0.455532,-0.554380,-0.101995,-0.082368,-0.485679,-0.099838,-0.164200
4,-0.668571,-0.137010,-0.554380,-0.101995,0.453689,-0.485679,-0.099838,-0.164200


## 17. Filtre de corrélation

Lorsque deux variables WoE sont fortement corrélées :

$$
|\rho(X_i,X_j)| > 0.75
$$

on conserve la variable présentant l'IV le plus élevé.

In [93]:
def correlation_filter(X, iv, threshold=0.75):
    cols = list(X.columns)
    corr = X[cols].corr().abs()
    keep = []

    for c in cols:
        remove = False

        for k in keep:
            value = corr.loc[c, k]

            if pd.notna(value) and value > threshold:
                if iv[c] > iv[k]:
                    keep.remove(k)
                    break
                else:
                    remove = True
                    break

        if not remove and c not in keep:
            keep.append(c)

    return keep


selected_corr = correlation_filter(
    W_dev[selected_iv],
    iv_table,
    CORR_MAX
)

selected_corr

['debt_ebitda',
 'roa',
 'leverage',
 'ebitda_margin',
 'utilization',
 'management_quality',
 'governance_score',
 'liquidity_ratio']

## 18. Filtre VIF

Le Variance Inflation Factor est utilisé pour contrôler la multicolinéarité :

$$
VIF_j = \frac{1}{1-R_j^2}
$$

On retire itérativement la variable ayant le VIF maximal lorsque le seuil est dépassé.

In [94]:
def vif_filter(X, threshold=5.0):
    cols = list(X.columns)

    if len(cols) <= 1:
        return cols, pd.Series(
            {c: 1.0 for c in cols},
            name="VIF"
        )

    while len(cols) > 1:
        Xv = sm.add_constant(X[cols], has_constant="add")

        vif_values = {
            c: variance_inflation_factor(
                Xv.values, i + 1
            )
            for i, c in enumerate(cols)
        }

        worst = max(vif_values, key=vif_values.get)
        max_vif = vif_values[worst]

        if np.isfinite(max_vif) and max_vif <= threshold:
            break

        cols.remove(worst)

    Xv = sm.add_constant(X[cols], has_constant="add")

    final_vif = pd.Series({
        c: variance_inflation_factor(
            Xv.values, i + 1
        )
        for i, c in enumerate(cols)
    }, name="VIF")

    return cols, final_vif


selected_vif, vif_table = vif_filter(
    W_dev[selected_corr],
    VIF_MAX
)

print("Variables après VIF :", selected_vif)
vif_table

Variables après VIF : ['debt_ebitda', 'roa', 'leverage', 'ebitda_margin', 'utilization', 'management_quality', 'governance_score', 'liquidity_ratio']


debt_ebitda          1.271272
roa                  3.312167
leverage             1.955000
ebitda_margin        2.906856
utilization          1.278699
management_quality   1.193704
governance_score     1.192314
liquidity_ratio      1.116054
Name: VIF, dtype: float64

## 19. Régression logistique

Le modèle est :

$$
logit(PD_i)
=
\beta_0
+
\sum_j \beta_j WoE_{ij}
$$

Avec la convention WoE retenue, on recherche généralement des coefficients positifs.

La PD est ensuite :

$$
PD_i =
\sigma(score_i)
=
\frac{1}{1+e^{-score_i}}
$$

In [95]:
def backward_logit(X, y, p_max=0.05, max_iter=30):
    cols = list(X.columns)

    while len(cols) > 1:
        model = sm.Logit(
            y,
            sm.add_constant(X[cols], has_constant="add")
        ).fit(disp=False)

        pvalues = model.pvalues.drop(
            labels=["const"], errors="ignore"
        )

        coeffs = model.params.drop(
            labels=["const"], errors="ignore"
        )

        bad_p = pvalues[pvalues > p_max]
        bad_sign = coeffs[coeffs <= 0]

        if bad_p.empty and bad_sign.empty:
            return model, cols

        candidates = list(
            set(bad_p.index).union(bad_sign.index)
        )

        if not candidates:
            return model, cols

        worst = max(
            candidates,
            key=lambda c: pvalues.get(c, 0)
        )

        cols.remove(worst)

    model = sm.Logit(
        y,
        sm.add_constant(X[cols], has_constant="add")
    ).fit(disp=False)

    return model, cols


final_model, final_vars = backward_logit(
    W_dev[selected_vif],
    dev_w[target].astype(int),
    P_VALUE_MAX
)

print("Variables finales :", final_vars)
print(final_model.summary())

Variables finales : ['debt_ebitda', 'ebitda_margin', 'utilization', 'management_quality', 'governance_score', 'liquidity_ratio']
                           Logit Regression Results                           
Dep. Variable:            default_dod   No. Observations:                17926
Model:                          Logit   Df Residuals:                    17919
Method:                           MLE   Df Model:                            6
Date:                Tue, 03 Nov 2026   Pseudo R-squ.:                 0.07114
Time:                        16:37:47   Log-Likelihood:                -3235.9
converged:                       True   LL-Null:                       -3483.7
Covariance Type:            nonrobust   LLR p-value:                7.162e-104
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -2.9657      0.037    -79.239      0.000   

## 20. Score et performance Development

On calcule :

- PD modèle ;
- AUC ;
- Gini ;
- KS ;
- Brier score.

Le Brier score est :

$$
Brier =
\frac{1}{N}
\sum_{i=1}^{N}
(PD_i-Y_i)^2
$$

In [96]:
def predict_pd(model, W, variables):
    X = sm.add_constant(
        W[variables],
        has_constant="add"
    )
    return model.predict(X)


dev_w["pd_model"] = predict_pd(
    final_model,
    W_dev,
    final_vars
)

oot_w["pd_model"] = predict_pd(
    final_model,
    W_oot,
    final_vars
)

perf_dev = performance_table(
    dev_w[target],
    dev_w["pd_model"]
)

perf_dev

AUC           0.710422
Gini          0.420844
KS            0.310441
Brier         0.044813
Observed_DR   0.048589
Mean_PD       0.048589
dtype: float64

## 21. Calibration Development

On compare les PD prédites et les taux de défaut observés par décile de risque.

In [97]:
cal_dev = calibration_table(
    dev_w[target],
    dev_w["pd_model"],
    n_bins=10
)

cal_dev

,bin,n,defaults,observed_dr,mean_pd,ratio_dr_pd
0,1,1795,12,0.006685,0.007154,0.934477
1,2,1791,27,0.015075,0.017091,0.882078
2,3,1792,40,0.022321,0.022663,0.984938
3,4,1793,53,0.029559,0.028077,1.052803
4,5,1793,72,0.040156,0.034080,1.178302
5,6,1792,72,0.040179,0.041012,0.979685
6,7,1792,81,0.045201,0.050443,0.896078
7,8,1793,118,0.065811,0.064008,1.028169
8,9,1792,152,0.084821,0.084890,0.999196
9,10,1793,244,0.136085,0.136483,0.997081


## 22. Validation OOT

L'OOT n'a pas été utilisé pour construire :

- les bins ;
- les WoE ;
- la sélection ;
- les coefficients.

Il fournit donc une mesure de robustesse temporelle.

In [98]:
perf_oot = performance_table(
    oot_w[target],
    oot_w["pd_model"]
)

pd.DataFrame({
    "Development": perf_dev,
    "OOT": perf_oot
})

,Development,OOT
AUC,0.710422,0.641830
Gini,0.420844,0.283660
KS,0.310441,0.215275
Brier,0.044813,0.041351
Observed_DR,0.048589,0.043628
Mean_PD,0.048589,0.044080


## 23. Performance par année et par taille

La performance doit rester acceptable sur les sous-populations importantes.

In [99]:
def yearly_performance(df, target_col, pd_col):
    rows = []

    for year, g in df.groupby("year"):
        auc = safe_auc(g[target_col], g[pd_col])

        rows.append({
            "year": year,
            "n": len(g),
            "defaults": g[target_col].sum(),
            "observed_dr": g[target_col].mean(),
            "mean_pd": g[pd_col].mean(),
            "AUC": auc,
            "Gini": gini_from_auc(auc) if pd.notna(auc) else np.nan,
            "KS": ks_stat(g[target_col], g[pd_col]),
        })

    return pd.DataFrame(rows)


year_input = pd.concat([
    dev_w[["obligor_id", "year", target, "pd_model"]],
    oot_w[["obligor_id", "year", target, "pd_model"]],
], ignore_index=True)

year_perf = yearly_performance(
    year_input,
    target,
    "pd_model"
)

year_perf

,year,n,defaults,observed_dr,mean_pd,AUC,Gini,KS
0,2015,2607,135,0.051784,0.050945,0.713314,0.426627,0.316199
1,2016,3365,179,0.053195,0.049193,0.723737,0.447474,0.368761
2,2017,3186,128,0.040176,0.048569,0.697338,0.394677,0.300140
3,2018,3058,132,0.043165,0.048125,0.697674,0.395348,0.299328
4,2019,2926,142,0.048530,0.047190,0.692128,0.384256,0.334158
5,2020,2784,155,0.055675,0.047655,0.731584,0.463169,0.354981
6,2021,2629,101,0.038418,0.044263,0.632161,0.264323,0.225318
7,2022,2528,121,0.047864,0.043858,0.631236,0.262471,0.204469
8,2023,2407,108,0.044869,0.044113,0.664246,0.328492,0.266497


In [100]:
def performance_by_group(df, group_col, target_col, pd_col):
    rows = []

    for group, g in df.groupby(group_col):
        auc = safe_auc(g[target_col], g[pd_col])

        rows.append({
            group_col: group,
            "n": len(g),
            "defaults": g[target_col].sum(),
            "observed_dr": g[target_col].mean(),
            "mean_pd": g[pd_col].mean(),
            "AUC": auc,
            "Gini": gini_from_auc(auc) if pd.notna(auc) else np.nan,
        })

    return pd.DataFrame(rows)


performance_by_group(
    oot_w,
    "size_class",
    target,
    "pd_model"
)

,size_class,n,defaults,observed_dr,mean_pd,AUC,Gini
0,Large,1138,54,0.047452,0.047100,0.663882,0.327764
1,Medium,2778,114,0.041037,0.043423,0.638212,0.276424
2,Small,3648,162,0.044408,0.043639,0.636433,0.272866


## 24. Refit sur l'historique complet

Après validation, le modèle peut être réestimé sur la période historique complète retenue pour la calibration.

Les transformations sont recalculées sur cet historique complet.

In [101]:
full_model_data = panel_model[
    panel_model["year"].isin(YEARS)
].copy()

full_wins = apply_winsor_limits(
    full_model_data,
    fit_winsor_limits(
        full_model_data,
        candidate_vars
    )
)

binner_full = WoEBinner(
    numeric_cols=candidate_vars
)

binner_full.fit(
    full_wins[candidate_vars],
    full_wins[target]
)

iv_full = (
    pd.Series(binner_full.iv)
    .sort_values(ascending=False)
)

selected_full_iv = iv_full[
    iv_full >= IV_THRESHOLD
].index.tolist()

W_full = binner_full.transform(
    full_wins[selected_full_iv]
)

selected_full_corr = correlation_filter(
    W_full,
    iv_full,
    CORR_MAX
)

selected_full_vif, vif_full = vif_filter(
    W_full[selected_full_corr],
    VIF_MAX
)

full_model, final_full_vars = backward_logit(
    W_full[selected_full_vif],
    full_wins[target].astype(int),
    P_VALUE_MAX
)

full_model_data["pd_pit_model"] = predict_pd(
    full_model,
    W_full,
    final_full_vars
)

print("Variables finales :", final_full_vars)

Variables finales : ['debt_ebitda', 'utilization', 'ebitda_margin', 'management_quality', 'governance_score', 'liquidity_ratio']


## 25. Calibration TTC — Long-Run Default Rate

Le long-run default rate est ici calculé comme :

$$
LRDR =
\frac{\sum_t Defaults_t}
{\sum_t Observations_t}
$$

Dans une implémentation bancaire réelle, la définition de l'unité d'exposition et la méthode de pondération doivent être documentées précisément.

In [102]:
annual_ttc = (
    full_model_data
    .groupby("year")
    .agg(
        n=("obligor_id", "size"),
        defaults=(target, "sum"),
        mean_pd=("pd_pit_model", "mean")
    )
)

annual_ttc["dr"] = (
    annual_ttc["defaults"]
    / annual_ttc["n"]
)

LRDR = (
    annual_ttc["defaults"].sum()
    / annual_ttc["n"].sum()
)

annual_ttc, LRDR

(         n  defaults  mean_pd       dr
 year                                  
 2015  2607       135 0.050662 0.051784
 2016  3365       179 0.048916 0.053195
 2017  3186       128 0.048566 0.040176
 2018  3058       132 0.047913 0.043165
 2019  2926       142 0.046966 0.048530
 2020  2784       155 0.047381 0.055675
 2021  2629       101 0.044376 0.038418
 2022  2528       121 0.043951 0.047864
 2023  2407       108 0.044023 0.044869,
 np.float64(0.04711651628089447))

## 26. Centrality shift

On cherche $\delta$ tel que :

$$
\frac{1}{N}
\sum_i
\sigma
\left(
logit(PD_i)+\delta
\right)
=
LRDR
$$

Le déplacement est effectué sur l'échelle logit.

In [103]:
def central_shift(
    pd_values,
    target_mean,
    lower=-20,
    upper=20,
    tolerance=1e-10,
    max_iter=200
):
    pd_values = np.asarray(pd_values)

    base_logit = logit(
        np.clip(pd_values, 1e-9, 1 - 1e-9)
    )

    def mean_shift(delta):
        return expit(base_logit + delta).mean()

    low, high = lower, upper

    for _ in range(max_iter):
        mid = (low + high) / 2
        value = mean_shift(mid)

        if abs(value - target_mean) < tolerance:
            return mid

        if value < target_mean:
            low = mid
        else:
            high = mid

    return (low + high) / 2


delta_ct = central_shift(
    full_model_data["pd_pit_model"],
    LRDR
)

full_model_data["pd_ttc"] = expit(
    logit(
        np.clip(
            full_model_data["pd_pit_model"],
            1e-9,
            1 - 1e-9
        )
    ) + delta_ct
)

print("Centrality shift :", delta_ct)
print("Mean PD TTC :", full_model_data["pd_ttc"].mean())
print("LRDR :", LRDR)

Centrality shift : 0.0
Mean PD TTC : 0.04711651628089448
LRDR : 0.04711651628089447


## 27. Master Scale Corporate

On utilise ici une Master Scale illustrative à 13 grades.

Les bornes sont des hypothèses de démonstration et ne constituent pas une Master Scale réglementaire.

In [104]:
MASTER_SCALE_UPPER_PD = np.array([
    0.0005,
    0.0010,
    0.0020,
    0.0040,
    0.0080,
    0.0150,
    0.0300,
    0.0600,
    0.1000,
    0.1600,
    0.2500,
    0.4000,
    1.0000,
])

MASTER_SCALE = pd.DataFrame({
    "grade": np.arange(1, 14),
    "upper_pd": MASTER_SCALE_UPPER_PD
})

MASTER_SCALE["grade_label"] = (
    "G" +
    MASTER_SCALE["grade"].astype(str).str.zfill(2)
)

MASTER_SCALE

,grade,upper_pd,grade_label
0,1,0.000500,G01
1,2,0.001000,G02
2,3,0.002000,G03
3,4,0.004000,G04
4,5,0.008000,G05
5,6,0.015000,G06
6,7,0.030000,G07
7,8,0.060000,G08
8,9,0.100000,G09
9,10,0.160000,G10


## 28. Attribution des grades

Pour une PD $p$, on attribue le premier grade dont la borne supérieure satisfait :

$$
p \leq UpperPD_g
$$

In [105]:
def assign_master_grade(pd_values, upper_bounds):
    grades = (
        np.searchsorted(
            upper_bounds,
            np.asarray(pd_values),
            side="left"
        ) + 1
    )

    return np.clip(
        grades,
        1,
        len(upper_bounds)
    )


full_model_data["grade"] = assign_master_grade(
    full_model_data["pd_ttc"],
    MASTER_SCALE_UPPER_PD
)

full_model_data["grade"].value_counts().sort_index()

grade
4      240
5      838
6     1415
7     7077
8     9351
9     4383
10    1918
11     268
Name: count, dtype: int64

## 29. LRA par grade

Pour chaque grade $g$, on calcule d'abord le taux de défaut annuel :

$$
DR_{g,t}
=
\frac{Defaults_{g,t}}
{N_{g,t}}
$$

puis :

$$
LRA_g =
\frac{1}{T_g}
\sum_t DR_{g,t}
$$

In [106]:
def lra_by_grade(df):
    annual = (
        df.groupby(["grade", "year"])
        .agg(
            n=(target, "size"),
            defaults=(target, "sum")
        )
        .reset_index()
    )

    annual["dr"] = (
        annual["defaults"]
        / annual["n"]
    )

    lra = annual.groupby("grade")["dr"].mean()
    counts = df.groupby("grade").size()

    return lra, counts, annual


lra, grade_n, grade_annual = lra_by_grade(
    full_model_data
)

pd.DataFrame({
    "LRA": lra,
    "n": grade_n
})

,LRA,n
grade,,
4,0.003367,240
5,0.002182,838
6,0.010043,1415
7,0.023797,7077
8,0.043632,9351
9,0.076316,4383
10,0.115560,1918
11,0.166740,268


## 30. Lissage log-linéaire

On ajuste :

$$
\ln(PD_g)=a+b\,g
$$

d'où :

$$
PD_g=e^{a+bg}
$$

Si la relation estimée n'est pas croissante, une isotonic regression croissante est utilisée comme solution de secours.

In [107]:
def smooth_loglinear(lra, counts, n_grades=13):
    valid = (
        lra.notna()
        & np.isfinite(lra)
        & (lra > 0)
    )

    if valid.sum() < 2:
        raise ValueError("Pas assez de grades observés.")

    x = lra.index[valid].astype(float).to_numpy()
    y = np.log(lra[valid].to_numpy())

    weights = np.sqrt(
        counts.reindex(lra.index[valid])
        .fillna(1)
        .to_numpy()
    )

    slope, intercept = np.polyfit(
        x, y, 1, w=weights
    )

    grades = np.arange(1, n_grades + 1)

    if slope > 0:
        values = np.exp(
            intercept + slope * grades
        )
    else:
        iso = IsotonicRegression(
            increasing=True,
            out_of_bounds="clip"
        )

        iso_values = iso.fit_transform(
            x,
            np.exp(y)
        )

        values = np.interp(
            grades,
            x,
            iso_values,
            left=iso_values[0],
            right=iso_values[-1]
        )

    return pd.Series(
        np.clip(values, 1e-6, 0.999999),
        index=grades,
        name="PD_smoothed"
    )


pd_smooth = smooth_loglinear(
    lra,
    grade_n
)

pd_smooth

1    0.000405
2    0.000782
3    0.001510
4    0.002914
5    0.005623
6    0.010852
7    0.020943
8    0.040418
9    0.078000
10   0.150529
11   0.290500
12   0.560624
13   0.999999
Name: PD_smoothed, dtype: float64

## 31. Margin of Conservatism

On illustre une décomposition :

$$
MoC =
MoC_A + MoC_B + MoC_C
$$

où :

- $MoC_A$ représente l'incertitude statistique ;
- $MoC_B$ représente une incertitude liée à la définition du défaut ;
- $MoC_C$ représente une incertitude modèle.

Les deux dernières composantes sont des hypothèses pédagogiques.

In [108]:
defaults_total = full_model_data[target].sum()
n_total = len(full_model_data)

se_lrdr = np.sqrt(
    LRDR * (1 - LRDR) / max(n_total, 1)
)

MOC_Z = norm.ppf(0.95)

moc_A = MOC_Z * se_lrdr / max(LRDR, 1e-9)
moc_B = 0.03
moc_C = 0.02

moc_total = moc_A + moc_B + moc_C

moc_table = pd.DataFrame({
    "component": [
        "MoC_A_statistical",
        "MoC_B_default_definition",
        "MoC_C_model_uncertainty",
        "MoC_total",
    ],
    "value": [
        moc_A,
        moc_B,
        moc_C,
        moc_total,
    ]
})

moc_table

,component,value
0,MoC_A_statistical,0.046331
1,MoC_B_default_definition,0.030000
2,MoC_C_model_uncertainty,0.020000
3,MoC_total,0.096331


## 32. PD finale par grade

On applique ici :

$$
PD^{MoC}_g
=
PD^{smooth}_g(1+MoC)
$$

puis un floor :

$$
PD^{final}_g
=
\max(PD^{MoC}_g,\;PD_{floor})
$$

Le traitement du MoC est volontairement simplifié.

In [109]:
PD_FLOOR = 0.0005

master_calibrated = MASTER_SCALE.copy()

master_calibrated["LRA_observed"] = (
    master_calibrated["grade"].map(lra)
)

master_calibrated["n"] = (
    master_calibrated["grade"].map(grade_n)
)

master_calibrated["PD_smoothed"] = (
    master_calibrated["grade"].map(pd_smooth)
)

master_calibrated["PD_MoC"] = np.clip(
    master_calibrated["PD_smoothed"]
    * (1 + moc_total),
    0.0001,
    0.9999
)

master_calibrated["PD_final"] = np.maximum(
    master_calibrated["PD_MoC"],
    PD_FLOOR
)

master_calibrated

,grade,upper_pd,grade_label,LRA_observed,n,PD_smoothed,PD_MoC,PD_final
0,1,0.000500,G01,NaN,NaN,0.000405,0.000444,0.000500
1,2,0.001000,G02,NaN,NaN,0.000782,0.000858,0.000858
2,3,0.002000,G03,NaN,NaN,0.001510,0.001655,0.001655
3,4,0.004000,G04,0.003367,240.000000,0.002914,0.003195,0.003195
4,5,0.008000,G05,0.002182,838.000000,0.005623,0.006165,0.006165
5,6,0.015000,G06,0.010043,"1,415.000000",0.010852,0.011898,0.011898
6,7,0.030000,G07,0.023797,"7,077.000000",0.020943,0.022961,0.022961
7,8,0.060000,G08,0.043632,"9,351.000000",0.040418,0.044311,0.044311
8,9,0.100000,G09,0.076316,"4,383.000000",0.078000,0.085514,0.085514
9,10,0.160000,G10,0.115560,"1,918.000000",0.150529,0.165030,0.165030


## 33. Overrides

Les overrides sont appliqués après le rating statistique.

Ils doivent être :

- documentés ;
- traçables ;
- bornés ;
- justifiés économiquement ;
- monitorés séparément.

Ici, un petit nombre d'overrides est simulé.

In [110]:
rng_override = np.random.default_rng(SEED + 1)

full_model_data["override_flag"] = (
    rng_override.random(len(full_model_data)) < 0.02
).astype(int)

full_model_data["override_direction"] = 0

mask = full_model_data["override_flag"].eq(1)

full_model_data.loc[mask, "override_direction"] = (
    rng_override.choice(
        [-1, 1],
        size=mask.sum()
    )
)

full_model_data["grade_final"] = np.clip(
    full_model_data["grade"]
    + full_model_data["override_direction"],
    1,
    13
)

pd_by_grade = (
    master_calibrated
    .set_index("grade")["PD_final"]
)

full_model_data["pd_final"] = (
    full_model_data["grade_final"]
    .map(pd_by_grade)
)

print("Taux d'override :", full_model_data["override_flag"].mean())

Taux d'override : 0.02047861906630051


## 34. Backtesting binomial

On teste :

$$
H_0: p = PD_{expected}
$$

contre :

$$
H_1: p > PD_{expected}
$$

Le test est ici unilatéral car on s'intéresse principalement au risque de sous-estimation du défaut.

In [111]:
def binomial_backtest(df):
    rows = []

    for grade, g in df.groupby("grade_final"):
        n = len(g)
        d = int(g[target].sum())
        p = float(g["pd_final"].mean())

        result = binomtest(
            d,
            n,
            p,
            alternative="greater"
        )

        rows.append({
            "grade": grade,
            "n": n,
            "defaults": d,
            "observed_dr": d / n,
            "expected_pd": p,
            "dr_pd_ratio": (d / n) / p if p > 0 else np.nan,
            "p_value": result.pvalue,
            "flag_5pct": result.pvalue < 0.05,
        })

    return pd.DataFrame(rows)


backtest_grade = binomial_backtest(
    full_model_data
)

backtest_grade

,grade,n,defaults,observed_dr,expected_pd,dr_pd_ratio,p_value,flag_5pct
0,3,4,0,0.000000,0.001655,0.000000,1.000000,False
1,4,243,1,0.004115,0.003195,1.288198,0.540456,False
2,5,834,3,0.003597,0.006165,0.583470,0.887532,False
3,6,1478,15,0.010149,0.011898,0.853012,0.765010,False
4,7,7028,168,0.023904,0.022961,1.041095,0.309266,False
5,8,9274,402,0.043347,0.044311,0.978243,0.680984,False
6,9,4419,339,0.076714,0.085514,0.897094,0.984068,False
7,10,1933,227,0.117434,0.165030,0.711592,1.000000,False
8,11,273,45,0.164835,0.318484,0.517561,1.000000,False
9,12,4,1,0.250000,0.614630,0.406749,0.977945,False


## 35. Jeffreys test

On utilise une distribution a posteriori :

$$
p \mid Data
\sim
Beta(d+0.5,\;n-d+0.5)
$$

et on examine :

$$
P(p > PD_{ref}\mid Data)
$$

In [112]:
def jeffreys_pvalue(d, n, p_ref):
    alpha = d + 0.5
    beta_param = n - d + 0.5

    if p_ref <= 0:
        return 1.0

    if p_ref >= 1:
        return 0.0

    return 1 - beta.cdf(
        p_ref,
        alpha,
        beta_param
    )


backtest_grade["jeffreys_pvalue"] = [
    jeffreys_pvalue(
        int(row["defaults"]),
        int(row["n"]),
        float(row["expected_pd"])
    )
    for _, row in backtest_grade.iterrows()
]

backtest_grade["jeffreys_flag_5pct"] = (
    backtest_grade["jeffreys_pvalue"] < 0.05
)

backtest_grade

,grade,n,defaults,observed_dr,expected_pd,dr_pd_ratio,p_value,flag_5pct,jeffreys_pvalue,jeffreys_flag_5pct
0,3,4,0,0.000000,0.001655,0.000000,1.000000,False,0.905458,False
1,4,243,1,0.004115,0.003195,1.288198,0.540456,False,0.669992,False
2,5,834,3,0.003597,0.006165,0.583470,0.887532,False,0.172219,False
3,6,1478,15,0.010149,0.011898,0.853012,0.765010,False,0.275690,False
4,7,7028,168,0.023904,0.022961,1.041095,0.309266,False,0.704451,False
5,8,9274,402,0.043347,0.044311,0.978243,0.680984,False,0.328144,False
6,9,4419,339,0.076714,0.085514,0.897094,0.984068,False,0.017075,True
7,10,1933,227,0.117434,0.165030,0.711592,1.000000,False,0.000000,True
8,11,273,45,0.164835,0.318484,0.517561,1.000000,False,0.000000,True
9,12,4,1,0.250000,0.614630,0.406749,0.977945,False,0.069140,False


## 36. Backtesting annuel

Le suivi temporel compare :

$$
Ratio_t =
\frac{DR_t}{PD_t}
$$

Un ratio durablement supérieur à 1 peut indiquer une sous-estimation de la PD.

In [113]:
def yearly_backtest(df):
    rows = []

    for year, g in df.groupby("year"):
        n = len(g)
        d = int(g[target].sum())
        p = float(g["pd_final"].mean())

        result = binomtest(
            d, n, p, alternative="greater"
        )

        rows.append({
            "year": year,
            "n": n,
            "defaults": d,
            "observed_dr": d / n,
            "mean_pd": p,
            "dr_pd_ratio": (d / n) / p if p > 0 else np.nan,
            "p_value": result.pvalue,
            "flag_5pct": result.pvalue < 0.05,
        })

    return pd.DataFrame(rows)


backtest_year = yearly_backtest(
    full_model_data
)

backtest_year

,year,n,defaults,observed_dr,mean_pd,dr_pd_ratio,p_value,flag_5pct
0,2015,2607,135,0.051784,0.058892,0.879304,0.945537,False
1,2016,3365,179,0.053195,0.056831,0.936011,0.828288,False
2,2017,3186,128,0.040176,0.056512,0.710929,0.999988,False
3,2018,3058,132,0.043165,0.055630,0.775934,0.999197,False
4,2019,2926,142,0.048530,0.053752,0.902860,0.903580,False
5,2020,2784,155,0.055675,0.054204,1.027148,0.377389,False
6,2021,2629,101,0.038418,0.050351,0.762998,0.998408,False
7,2022,2528,121,0.047864,0.050247,0.952566,0.720798,False
8,2023,2407,108,0.044869,0.049690,0.902988,0.872821,False


## 37. Test de Vasicek simplifié

Pour une fréquence de défaut observée $DR$ et une PD attendue $PD$, une transformation inspirée de Vasicek est utilisée.

Le paramètre $\rho$ est ici illustratif :

$$
Z =
\frac{
\sqrt{1-\rho}\Phi^{-1}(DR)
-
\Phi^{-1}(PD)
}
{\sqrt{\rho}}
$$

puis :

$$
p = 1-\Phi(Z)
$$

In [114]:
def vasicek_pvalue(observed_dr, expected_pd, rho=0.15):
    observed_dr = np.clip(observed_dr, 1e-8, 1 - 1e-8)
    expected_pd = np.clip(expected_pd, 1e-8, 1 - 1e-8)

    z = (
        np.sqrt(1 - rho) * norm.ppf(observed_dr)
        - norm.ppf(expected_pd)
    ) / np.sqrt(rho)

    return 1 - norm.cdf(z)


vasicek = []

for _, row in backtest_year.iterrows():
    vasicek.append({
        "year": row["year"],
        "vasicek_pvalue": vasicek_pvalue(
            row["observed_dr"],
            row["mean_pd"]
        )
    })

vasicek_df = pd.DataFrame(vasicek)

backtest_year.merge(
    vasicek_df,
    on="year"
)

,year,n,defaults,observed_dr,mean_pd,dr_pd_ratio,p_value,flag_5pct,vasicek_pvalue
0,2015,2607,135,0.051784,0.058892,0.879304,0.945537,False,0.434996
1,2016,3365,179,0.053195,0.056831,0.936011,0.828288,False,0.404795
2,2017,3186,128,0.040176,0.056512,0.710929,0.999988,False,0.528227
3,2018,3058,132,0.043165,0.055630,0.775934,0.999197,False,0.488333
4,2019,2926,142,0.048530,0.053752,0.902860,0.903580,False,0.418438
5,2020,2784,155,0.055675,0.054204,1.027148,0.377389,False,0.361285
6,2021,2629,101,0.038418,0.050351,0.762998,0.998408,False,0.489495
7,2022,2528,121,0.047864,0.050247,0.952566,0.720798,False,0.391615
8,2023,2407,108,0.044869,0.049690,0.902988,0.872821,False,0.414634


## 38. PSI

Le PSI est calculé par :

$$
PSI =
\sum_i
(A_i-E_i)
\ln
\left(
\frac{A_i}{E_i}
\right)
$$

où $A_i$ représente la proportion actuelle et $E_i$ la proportion de référence.

Les seuils usuels sont indicatifs :

- $PSI<0.10$ : faible changement ;
- $0.10\le PSI<0.25$ : surveillance ;
- $PSI\ge0.25$ : changement important.

In [115]:

import numpy as np
import pandas as pd

def psi(expected, actual, bins=None, eps=1e-6):
    """
    Population Stability Index (PSI).

    expected : population de référence (Development)
    actual   : population comparée (OOT)
    bins     : bornes communes aux deux populations
    eps      : correction pour éviter log(0)
    """

    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    if len(expected) == 0 or len(actual) == 0:
        return np.nan

    # Création des classes si elles ne sont pas fournies
    if bins is None:
        unique_values = np.sort(expected.unique())

        if len(unique_values) <= 1:
            return 0.0

        bins = np.unique(
            np.quantile(
                expected,
                np.linspace(0, 1, min(10, len(unique_values)) + 1)
            )
        )

        if len(bins) <= 2:
            return 0.0

        bins[0] = -np.inf
        bins[-1] = np.inf

    # Affectation des observations aux mêmes classes
    e_cat = pd.cut(
        expected,
        bins=bins,
        include_lowest=True
    )

    a_cat = pd.cut(
        actual,
        bins=bins,
        include_lowest=True
    )

    # Fréquences relatives
    e = e_cat.value_counts(sort=False) / len(expected)
    a = a_cat.value_counts(sort=False) / len(actual)

    # Alignement des catégories
    a = a.reindex(e.index, fill_value=0)

    # Correction des fréquences nulles
    e = np.maximum(e.to_numpy(dtype=float), eps)
    a = np.maximum(a.to_numpy(dtype=float), eps)

    # Renormalisation
    e = e / e.sum()
    a = a / a.sum()

    # Calcul du PSI
    psi_value = np.sum(
        (a - e) * np.log(a / e)
    )

    return float(psi_value)

    
grade_bins = np.arange(0.5, 14.5, 1)

dev_grades = full_model_data.loc[
    full_model_data["year"].isin(DEVELOPMENT_YEARS),
    "grade"
]

oot_grades = full_model_data.loc[
    full_model_data["year"].isin(OOT_YEARS),
    "grade"
]

psi_grade = psi(
    dev_grades,
    oot_grades,
    grade_bins
)

print("PSI grades :", psi_grade)

PSI grades : 0.01840560712877605


## 39. PSI des variables d'entrée

Le PSI peut être calculé sur les variables candidates afin de détecter des changements de population entre le développement et la production.

In [116]:
psi_results = []

for c in candidate_vars:
    dev_values = full_wins.loc[
        full_wins["year"].isin(DEVELOPMENT_YEARS),
        c
    ]

    oot_values = full_wins.loc[
        full_wins["year"].isin(OOT_YEARS),
        c
    ]

    non_null = dev_values.dropna()

    if non_null.nunique() < 2:
        continue

    bins = (
        non_null
        .quantile(np.linspace(0, 1, 11))
        .drop_duplicates()
        .to_numpy()
    )
    bins = np.array(bins, dtype=float, copy=True)
    if len(bins) < 2:
        continue

    bins[0] = -np.inf
    bins[-1] = np.inf

    psi_results.append({
        "variable": c,
        "PSI": psi(
            dev_values,
            oot_values,
            bins
        )
    })

psi_table = (
    pd.DataFrame(psi_results)
    .sort_values("PSI", ascending=False)
)

psi_table

,variable,PSI
3,roa,0.018044
1,debt_ebitda,0.017866
0,leverage,0.010346
2,ebitda_margin,0.008754
6,utilization,0.004616
7,management_quality,0.003346
8,governance_score,0.003319
4,liquidity_ratio,0.001495
5,cash_ratio,0.001287


## 40. Portefeuille 2023

On extrait maintenant le portefeuille de production 2023.

Le portefeuille reçoit :

- PD modèle ;
- grade statistique ;
- grade final ;
- PD finale.

In [117]:
portfolio_2023 = full_model_data[
    full_model_data["year"] == TARGET_YEAR
].copy()

portfolio_2023[
    [
        "obligor_id",
        "sector",
        "size_class",
        "grade",
        "grade_final",
        "pd_final"
    ]
].head(20)

,obligor_id,sector,size_class,grade,grade_final,pd_final
8,1,Retail,Small,7,7,0.022961
16,2,Services,Small,10,10,0.165030
34,4,Tech,Medium,7,7,0.022961
51,6,Construction,Large,8,8,0.044311
60,7,Industry,Medium,8,8,0.044311
69,8,Energy,Small,8,8,0.044311
87,10,Energy,Small,8,8,0.044311
96,11,Services,Medium,6,6,0.011898
104,12,Industry,Medium,8,8,0.044311
121,14,Industry,Large,7,7,0.022961


## 41. Distribution finale

On examine les statistiques de la PD finale.

In [118]:
portfolio_2023["pd_final"].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

count   2,407.000000
mean        0.049690
std         0.043387
min         0.001655
1%          0.006165
5%          0.011898
25%         0.022961
50%         0.044311
75%         0.044311
95%         0.165030
99%         0.165030
max         0.318484
Name: pd_final, dtype: float64

## 42. Calibration finale par grade

In [119]:
final_calibration = (
    full_model_data
    .groupby("grade_final")
    .agg(
        n=("obligor_id", "size"),
        defaults=(target, "sum"),
        observed_dr=(target, "mean"),
        mean_pd=("pd_final", "mean")
    )
    .reset_index()
)

final_calibration["dr_pd_ratio"] = (
    final_calibration["observed_dr"]
    / final_calibration["mean_pd"].replace(0, np.nan)
)

final_calibration

,grade_final,n,defaults,observed_dr,mean_pd,dr_pd_ratio
0,3,4,0,0.000000,0.001655,0.000000
1,4,243,1,0.004115,0.003195,1.288198
2,5,834,3,0.003597,0.006165,0.583470
3,6,1478,15,0.010149,0.011898,0.853012
4,7,7028,168,0.023904,0.022961,1.041095
5,8,9274,402,0.043347,0.044311,0.978243
6,9,4419,339,0.076714,0.085514,0.897094
7,10,1933,227,0.117434,0.165030,0.711592
8,11,273,45,0.164835,0.318484,0.517561
9,12,4,1,0.250000,0.614630,0.406749


## 43. Contrôle de monotonicité

On vérifie :

$$
PD_{G1}
\leq
PD_{G2}
\leq
...
\leq
PD_{G13}
$$

In [120]:
master_scale_monotone = (
    master_calibrated["PD_final"]
    .diff()
    .dropna()
    .ge(0)
    .all()
)

print("Master Scale monotone :", master_scale_monotone)

Master Scale monotone : True


## 44. Contrôles qualité finaux

Les contrôles vérifient notamment :

- PD strictement positive ;
- PD < 1 ;
- grades valides ;
- absence de PD manquante ;
- monotonicité de la Master Scale ;
- centralité TTC cohérente.

In [121]:
quality_checks = {
    "pd_positive": bool(
        (full_model_data["pd_final"] > 0).all()
    ),
    "pd_below_one": bool(
        (full_model_data["pd_final"] < 1).all()
    ),
    "grade_valid": bool(
        full_model_data["grade_final"].between(1, 13).all()
    ),
    "master_scale_monotone": bool(
        master_scale_monotone
    ),
    "no_final_pd_missing": bool(
        full_model_data["pd_final"].notna().all()
    ),
    "central_tendency_close": bool(
        abs(
            full_model_data["pd_ttc"].mean() - LRDR
        ) < 1e-6
    ),
}

pd.Series(quality_checks, name="passed")

pd_positive               True
pd_below_one              True
grade_valid               True
master_scale_monotone     True
no_final_pd_missing       True
central_tendency_close    True
Name: passed, dtype: bool

## 45. Synthèse du modèle

Cette synthèse regroupe les principaux indicateurs de performance, calibration et stabilité.

In [122]:
final_auc = safe_auc(
    full_model_data[target],
    full_model_data["grade_final"]
)

summary = pd.DataFrame({
    "Metric": [
        "Observations",
        "Defaults",
        "Observed Default Rate",
        "LRDR",
        "AUC Development",
        "Gini Development",
        "AUC OOT",
        "Gini OOT",
        "KS Development",
        "PSI Grades",
        "MoC Total",
        "PD Floor",
        "Centrality Shift",
        "Final Grade AUC",
        "Final Grade Gini",
    ],
    "Value": [
        len(full_model_data),
        int(full_model_data[target].sum()),
        full_model_data[target].mean(),
        LRDR,
        perf_dev["AUC"],
        perf_dev["Gini"],
        perf_oot["AUC"],
        perf_oot["Gini"],
        perf_dev["KS"],
        psi_grade,
        moc_total,
        PD_FLOOR,
        delta_ct,
        final_auc,
        gini_from_auc(final_auc),
    ]
})

summary

,Metric,Value
0,Observations,"25,490.000000"
1,Defaults,"1,201.000000"
2,Observed Default Rate,0.047117
3,LRDR,0.047117
4,AUC Development,0.710422
5,Gini Development,0.420844
6,AUC OOT,0.641830
7,Gini OOT,0.283660
8,KS Development,0.310441
9,PSI Grades,0.018406


## 46. Inventaire final des variables

Le tableau constitue un livrable important de documentation du modèle.

In [123]:
final_variable_inventory = pd.DataFrame({
    "variable": final_full_vars,
    "IV": [
        iv_full.get(v, np.nan)
        for v in final_full_vars
    ],
    "VIF": [
        vif_full.get(v, np.nan)
        for v in final_full_vars
    ],
    "coefficient": [
        full_model.params.get(v, np.nan)
        for v in final_full_vars
    ],
    "p_value": [
        full_model.pvalues.get(v, np.nan)
        for v in final_full_vars
    ],
})

final_variable_inventory

,variable,IV,VIF,coefficient,p_value
0,debt_ebitda,0.391272,1.439896,0.771326,0.000000
1,utilization,0.096668,1.270675,0.462121,0.000010
2,ebitda_margin,0.093107,2.694709,0.692861,0.000000
3,management_quality,0.086138,1.222159,0.850758,0.000000
4,governance_score,0.060417,1.220989,0.568770,0.000025
5,liquidity_ratio,0.039162,1.105599,0.338761,0.030544


## 47. Fonction de scoring de production

La fonction suivante matérialise le moteur de scoring :

$$
X_{raw}
\rightarrow
Bins
\rightarrow
WoE
\rightarrow
Score
\rightarrow
PD
\rightarrow
Grade
\rightarrow
PD_{final}
$$

In [124]:
def score_corporate_portfolio(
    df_raw,
    binner,
    model,
    model_variables,
    selected_input_variables,
    grade_upper_bounds,
    pd_by_grade,
    pd_floor=0.0005,
):
    df = df_raw.copy()

    W = binner.transform(
        df[selected_input_variables]
    )

    df["pd_model"] = predict_pd(
        model,
        W,
        model_variables
    )

    df["grade"] = assign_master_grade(
        df["pd_model"],
        grade_upper_bounds
    )

    df["pd_final"] = (
        df["grade"].map(pd_by_grade)
    )

    df["pd_final"] = np.maximum(
        df["pd_final"],
        pd_floor
    )

    return df

## 48. Test du moteur de scoring

On vérifie que le moteur de production restitue les mêmes PD que le calcul précédent.

In [125]:
production_2023 = score_corporate_portfolio(
    portfolio_2023,
    binner_full,
    full_model,
    final_full_vars,
    selected_full_iv,
    MASTER_SCALE_UPPER_PD,
    pd_by_grade,
    PD_FLOOR
)

comparison = pd.DataFrame({
    "pd_previous": portfolio_2023["pd_final"].to_numpy(),
    "pd_production": production_2023["pd_final"].to_numpy(),
})

comparison["difference"] = (
    comparison["pd_previous"]
    - comparison["pd_production"]
)

print(
    "Différence absolue maximale :",
    comparison["difference"].abs().max()
)

Différence absolue maximale : 0.07951592952239703


## 49. Exports

Les principaux livrables sont exportés en CSV :

- Master Scale ;
- variables du modèle ;
- backtesting ;
- monitoring ;
- portefeuille scoré 2023.

In [126]:
master_calibrated.to_csv(
    "corporate_master_scale.csv",
    index=False
)

final_variable_inventory.to_csv(
    "corporate_model_variables.csv",
    index=False
)

backtest_grade.to_csv(
    "corporate_backtesting_grade.csv",
    index=False
)

backtest_year.to_csv(
    "corporate_backtesting_year.csv",
    index=False
)

psi_table.to_csv(
    "corporate_psi.csv",
    index=False
)

production_2023.to_csv(
    "corporate_scored_portfolio_2023.csv",
    index=False
)

summary.to_csv(
    "corporate_model_summary.csv",
    index=False
)

print("Exports terminés.")

Exports terminés.


# 50. Rapport final

Le pipeline complet est :

$$
Data
\rightarrow
DoD
\rightarrow
Development/OOT
\rightarrow
Binning
\rightarrow
WoE/IV
\rightarrow
Correlation/VIF
\rightarrow
Logit
\rightarrow
Validation
\rightarrow
TTC
\rightarrow
MasterScale
\rightarrow
MoC
\rightarrow
Floor
\rightarrow
Overrides
\rightarrow
Backtesting
\rightarrow
PSI
\rightarrow
Production
$$

La sortie principale du modèle est une **PD Corporate finale par obligor**, accompagnée d'un **grade Corporate** et des éléments nécessaires au monitoring.

> Pour une utilisation bancaire réelle, les données simulées doivent être remplacées par les données historiques de la banque et les hypothèses réglementaires illustratives doivent être remplacées par la méthodologie approuvée.